# Day 2:
**Joins & Architecture:** `INNER`, `LEFT`, `RIGHT`, `FULL OUTER`, Self-joins, and `CROSS JOIN`. Query structuring using Common Table Expressions (CTEs with `WITH`).

In [3]:
# Imports:
import sqlite3
import pandas as pd

# DB Connection:
conn= sqlite3.connect('/home/shail/interview-prep/01_SQL/oracle_hr.db')

## SQL Joins & Architecture

**Explanation:** Relational databases split information into separate tables to avoid duplicating data. Joins are the mechanisms used to stitch that separated data back together into a complete picture based on a shared column (usually an ID).

**The Core Join Types:**
*   **INNER JOIN:** Strict match. Returns rows ONLY when there is a match in *both* tables. 
*   **LEFT JOIN:** Generous host. Returns *all* rows from the left table, plus any matched rows from the right. If there is no match on the right, it fills those columns with `NULL`.
*   **RIGHT JOIN:** The exact inverse of a LEFT JOIN. Returns *all* rows from the right table.
*   **FULL OUTER JOIN:** Gives everything. Returns all rows from both tables, inserting `NULL` wherever there isn't a match.

**Architectural Joins:**
*   **SELF-JOIN:** Joining a table to itself. Used for hierarchical data within the same table (e.g., finding an employee's manager when both are in the `employees` table).
*   **CROSS JOIN:** Combines everything to find all possible combinations (Cartesian product). It does not use an `ON` clause to bridge data. 

### Example Code Syntax

```sql
-- 1. INNER JOIN (Standard)
SELECT e.first_name, d.department_name
FROM employees e
INNER JOIN departments d 
    ON e.department_id = d.department_id;

-- 2. LEFT JOIN
SELECT e.first_name, d.department_name
FROM employees e
LEFT JOIN departments d 
    ON e.department_id = d.department_id;

-- 3. SELF-JOIN (Employee to Manager)
SELECT emp.first_name AS employee, mgr.first_name AS manager
FROM employees emp
INNER JOIN employees mgr 
    ON emp.manager_id = mgr.employee_id;

-- 4. CROSS JOIN (All combinations)
SELECT s.shirt_color, p.pants_color
FROM shirts s
CROSS JOIN pants p;

### Question 1:
In our HR database, we have:

An employees table (columns: employee_id, first_name, job_id)

A jobs table (columns: job_id, job_title)

Your Task: Write an INNER JOIN to return a list of every employee's first_name alongside their actual job_title.

In [8]:
pd.read_sql_query(sql= """
select e.first_name, j.job_title
from employees e
join jobs j
on (e.job_id = j.job_id);
""", con= conn)

,first_name,job_title
0,Steven,President
1,Neena,Administration Vice President
2,Lex,Administration Vice President
3,Alexander,Programmer
4,Bruce,Programmer
...,...,...
102,Pat,Marketing Representative
103,Susan,Human Resources Representative
104,Hermann,Public Relations Representative
105,Shelley,Accounting Manager


## Question 2:
Your Task: Write a query that finds employees who work in the same department.

Select the first_name of the first employee, the first_name of the second employee, and their shared department_id.

You will need to join the employees table to itself. Use e1 and e2 as your aliases.

Bridge them using the department_id.

In [16]:
pd.read_sql_query(sql= """
select e1.first_name, e2.first_name, e1.department_id
from employees e1
join employees e2
on (e1.department_id= e2.department_id)
where e1.employee_id <> e2.employee_id;
""", con= conn)

,first_name,first_name,department_id
0,Steven,Lex,90
1,Steven,Neena,90
2,Neena,Lex,90
3,Neena,Steven,90
4,Lex,Neena,90
...,...,...,...
3187,Douglas,Winston,50
3188,Michael,Pat,20
3189,Pat,Michael,20
3190,Shelley,William,110


## Common Table Expressions (CTEs)

**Explanation:** A CTE (Common Table Expression) creates a temporary, named result set that exists only for the duration of a single query.

**Core Intuition:** Think of a CTE as doing "prep work" before cooking. Instead of writing messy, deeply nested subqueries (queries stuffed inside other queries), you can isolate intermediate steps at the top of your script. This makes complex data transformations modular, readable, and easier to debug.

**Pros and Cons:**
*   **Pros:** Massively improves readability of complex queries; allows you to reference the same temporary dataset multiple times in the main query.
*   **Cons:** They are generally not materialized (stored on disk), meaning if you reference a complex CTE multiple times, the database might recalculate it from scratch each time, which can impact performance.

### Example Code Syntax

```sql
-- Define the CTE using WITH
WITH Sales_Team AS (
    SELECT employee_id, first_name
    FROM employees
    WHERE department_id = 80
)

-- Query the CTE as if it were a real table
SELECT * 
FROM Sales_Team;

### Question 1:
Your Task:

Use the WITH clause to create a CTE named Sales_Team.

Inside that CTE, select employee_id and first_name from the employees table, but only for employees where the department_id = 80 (the sales department).

Underneath your CTE, write a simple SELECT statement to pull all columns (*) from your new Sales_Team CTE.

In [10]:
pd.read_sql_query(sql="""
with sales_team as (
select employee_id, first_name
from employees
where department_id= 80)
select * from sales_team;
""", con= conn)

,employee_id,first_name
0,145,John
1,146,Karen
2,147,Alberto
3,148,Gerald
4,149,Eleni
5,150,Sean
6,151,David
7,152,Peter
8,153,Christopher
9,154,Nanette


## Python (Two-Pointer Technique & Sliding Window)

These are foundational strategies for solving problems with arrays (lists) or strings in Python. 

### The Core Intuition: Two-Pointer Technique
Imagine you have a long, straight row of numbered boxes, and you are trying to find two specific boxes. 
*   A naive approach is to start at box 1, then check every other box. Then start at box 2, and check every other box. This is slow.
*   The **Two-Pointer Technique** involves placing one finger (pointer) at the start of the row, and one finger at the end of the row. You then move your fingers toward each other based on certain conditions. It is vastly more efficient for searching or comparing.

### The Core Intuition: Sliding Window
Imagine viewing a long landscape painting through a small, square window cut out of a piece of cardboard.
*   You don't look at the whole painting at once. You look at a small section (the window), and then you slide the cardboard slightly to the right to see the next section.
*   The **Sliding Window** technique is used when you need to analyze a *continuous subset* of data (like finding the largest sum of any 3 consecutive numbers in a list). The "window" has a fixed size, and it slides across the data.

### The Syntax Walkthrough (Two-Pointer)

Let's look at a classic Two-Pointer example: reversing a list of characters in place.

'''python
def reverse_string(char_list):
    # 1. Set our pointers
    left = 0
    right = len(char_list) - 1
    
    # 2. Move them toward each other
    while left < right:
        # Swap the characters
        temp = char_list[left]
        char_list[left] = char_list[right]
        char_list[right] = temp
        
        # 3. Step inward
        left += 1
        right -= 1
        
    return char_list

my_list = ['A', 'B', 'C', 'D']

print(reverse_string(my_list)) 

Output: ['D', 'C', 'B', 'A']

'''

**Step-by-Step Breakdown:**
1.  We define `left` at the start (`0`) and `right` at the end (`len - 1`).
2.  We use a `while` loop that runs as long as `left` is strictly less than `right`.
3.  Inside the loop, we temporarily store the left value, overwrite the left with the right, and then overwrite the right with the stored temp value.
4.  Crucially, we increment `left` and decrement `right` so they step toward the middle.

### Question 1:
Task: Write a Python function that uses two pointers to determine if a string is a palindrome (reads the same forwards and backwards, like "racecar").

In [22]:
test_words = ["racecar", "python", "level"]

def is_palindrome(word: str):
    left= 0
    right= len(word)- 1

    while left <= right:
        if word[left] == word[right]:
            left+= 1
            right -= 1
            continue

        else:
            print(f'"{word}" is Not Palindrome.')
            break
    else:
        print(f'"{word} is Palindrome.')

for w in test_words:
    is_palindrome(word= w)

"racecar is Palindrome.
"python" is Not Palindrome.
"level is Palindrome.
